# Hands On with Advanced AI & Emerging Applications

# Exploring Open-weight models

In [ ]:
!pip install -q transformers ipywidgets


Log in to access gated models. Replace `<huggingface_token>` with the token provided by the instuctor. You can get your own token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
!hf auth login --token <huggingface_token>

Load the tokenizer and model separately:
- **Tokenizer**: Converts text into numerical tokens that the model can process. It's a lightweight component that handles text preprocessing.
- **Model**: The actual neural network with billions of parameters. We load it in `bfloat16` precision (2 bytes per parameter) to reduce memory usage and automatically place it on GPU using `device_map="cuda"`.

Hugging Face separates tokenizers and models because they serve different purposes.
The tokenizer converts text into token IDs on the CPU and has no neural weights, while the model performs neural inference on those token IDs, often on a GPU.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct",
                                              torch_dtype=torch.bfloat16,
                                              device_map="cuda")

View the complete model structure including all layers and components.

In [ ]:
model


Examine important configuration details:
- **Layers**: Number of transformer blocks
- **Hidden size**: Dimension of internal representations. Embeddings dimensions
- **Attention heads**: Parallel attention mechanisms, per Transformer block,  that help the model focus on different aspects of the input
- **KV heads**: Key-Value heads used in Grouped Query Attention (fewer than query heads for efficiency)
- **Max context**: Maximum input sequence length, also known as context lenght
- **Vocab size**: Number of tokens the model understands

In [ ]:
config = model.config

print("Architecture:", config.architectures)
print("Layers:", config.num_hidden_layers)
print("Hidden size:", config.hidden_size)
print("Attention heads:", config.num_attention_heads)
print("KV heads:", getattr(config, "num_key_value_heads", "N/A"))
print("Max context:", config.max_position_embeddings)
print("Vocab size:", config.vocab_size)
print("Dtype:", config.torch_dtype)


The embedding layer converts tokens (words/subwords) into dense vector representations that the model can process.

In [ ]:
embedding = model.get_input_embeddings()

print("Embedding shape:", embedding.weight.shape)
print("Embedding dtype:", embedding.weight.dtype)

Count total and trainable parameters to understand the model's capacity and memory requirements.

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


Check GPU availability and memory usage. As we are only loading the model the GPU memory consumption is roughly the #Parameters * Parameters Precision, 1.235B x 2 bytes ~ 2.5 GB

In [ ]:
!nvidia-smi

Each layer contains self-attention and feed-forward components that transform the input progressively.

In [ ]:
layers = model.model.layers

print("Number of layers:", len(layers))
print(layers[0])


The attention mechanism uses three learned projections:
- **Q (Query)**: What we're looking for
- **K (Key)**: What each position offers
- **V (Value)**: The actual information to retrieve



In [ ]:
layer = model.model.layers[0]
attn = layer.self_attn

print("Q:", attn.q_proj.weight.shape)
print("K:", attn.k_proj.weight.shape)
print("V:", attn.v_proj.weight.shape)

View the actual learned parameters from the Query projection. These weights are trained during model training.

In [ ]:
attn.q_proj.weight[0, :10]

## Text Generation
Generate text using the model. The chat template formats the input properly, and we decode only the newly generated tokens.

In [ ]:
messages = [
    {"role": "user", "content": "Why the sky is bkue?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))